# Module 2: OIDC SSO Validation

## Overview

This notebook validates OIDC SSO configuration for your LangSmith deployment. It assumes Module 1 is complete (working deployment with DNS/TLS/Ingress).

**Prerequisites:**
- Module 1 deployment is healthy and accessible
- DNS configured and resolving correctly
- TLS certificate valid and trusted
- Ingress configured and working
- IdP team has provided OIDC configuration values

## What We'll Validate

1. ✅ Environment configuration (OIDC settings, redacted)
2. ✅ Preflight checks (tools, kubectl, namespace, Helm release)
3. ✅ Current auth configuration (without leaking secrets)
4. ✅ Ingress/TLS preconditions (domain, HTTPS)
5. ✅ OIDC settings validation (issuer, redirect URI, claims)
6. ✅ Deployment verification (pods, logs, endpoints)
7. ✅ Failure drills guidance (optional, opt-in)
8. ✅ Support bundle pointers

**Estimated time:** 30-45 minutes

**Important:** This notebook never prints secrets. All sensitive values are redacted.


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path so we can import shared as a package
possible_paths = [
    Path.cwd().parent,  # If cwd is module-2, go up one level to notebooks
    Path.cwd(),  # If cwd is already notebooks
    Path.cwd() / "notebooks",  # If cwd is workspace root
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

# Add notebooks directory to path so 'shared' can be imported as a package
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])
print(f"\nArtifacts directory: {artifacts_dir}")


## 1. Configuration

Load and validate OIDC configuration from environment variables. All secrets are redacted in output.


In [ ]:
import os
import json
from shared._validation import require_env, print_config, redact, ok, warn

# Required OIDC configuration variables
required_vars = [
    "NAMESPACE",
    "OIDC_ISSUER",
    "OIDC_CLIENT_ID",
    "OIDC_CLIENT_SECRET",
    "OIDC_REDIRECT_URI",
    "LANGSMITH_DOMAIN",  # Domain where LangSmith is accessible
]

# Optional but recommended
optional_vars = [
    "OIDC_SCOPES",
    "OIDC_EMAIL_CLAIM",
    "OIDC_NAME_CLAIM",
    "OIDC_GROUPS_CLAIM",
]

print("### Loading OIDC Configuration\n")

# Load required variables
config = {}
missing = []

for var in required_vars:
    value = os.environ.get(var, "").strip()
    if not value:
        missing.append(var)
    config[var] = value

if missing:
    raise RuntimeError(f"❌ Missing required environment variables: {', '.join(missing)}\n"
                      f"💡 Copy env-samples/oidc.env.example to your .env file and fill in values")

# Load optional variables
for var in optional_vars:
    config[var] = os.environ.get(var, "").strip()

# Set defaults for optional variables
if not config.get("OIDC_SCOPES"):
    config["OIDC_SCOPES"] = "openid,email,profile,groups"
if not config.get("OIDC_EMAIL_CLAIM"):
    config["OIDC_EMAIL_CLAIM"] = "email"
if not config.get("OIDC_NAME_CLAIM"):
    config["OIDC_NAME_CLAIM"] = "name"
if not config.get("OIDC_GROUPS_CLAIM"):
    config["OIDC_GROUPS_CLAIM"] = "groups"

# Print configuration (redacted)
print_config(config, redact_keys={"OIDC_CLIENT_SECRET"})

ok("Configuration loaded (secrets redacted)")

# Validate redirect URI format
redirect_uri = config["OIDC_REDIRECT_URI"]
if not redirect_uri.startswith("https://"):
    warn("Redirect URI should use HTTPS in production")
if not redirect_uri.endswith("/auth/callback"):
    warn("Redirect URI should typically end with /auth/callback")
    print(f"   Current: {redirect_uri}")

# Validate domain matches redirect URI
domain = config["LANGSMITH_DOMAIN"]
if domain not in redirect_uri:
    warn(f"Domain '{domain}' not found in redirect URI '{redirect_uri}'")
    print("   💡 Redirect URI domain should match LANGSMITH_DOMAIN")

print(f"\n💡 Verify these values match your IdP configuration:")
print(f"   - Issuer: {config['OIDC_ISSUER']}")
print(f"   - Client ID: {config['OIDC_CLIENT_ID']}")
print(f"   - Redirect URI: {redirect_uri} (must be whitelisted in IdP)")


## 2. Preflight Checks

Verify tools, kubectl context, namespace, and Helm release exist.


In [ ]:
from shared._validation import ok, warn
from shared._k8s_helpers import require_namespace, namespace_exists
from shared._shell import run
from shared._cloud_helpers import get_cloud_provider, get_region, configure_kubectl

provider = get_cloud_provider()
region = get_region()
namespace = config["NAMESPACE"]

print("### Preflight Checks\n")

# Check kubectl is available
print("1. Checking kubectl...")
result = run(["kubectl", "version", "--client", "--short"], check=False, stream=False)
if result.returncode == 0:
    ok("kubectl is available")
    print(f"   {result.stdout.strip()}")
else:
    raise RuntimeError("❌ kubectl is not available or not working")

# Check kubectl context
print("\n2. Checking kubectl context...")
result = run(["kubectl", "config", "current-context"], check=False, stream=False)
if result.returncode == 0:
    context = result.stdout.strip()
    ok(f"Current context: {context}")
else:
    warn("Could not determine kubectl context")
    print("   💡 Run: kubectl config get-contexts")

# Check namespace exists
print(f"\n3. Checking namespace '{namespace}'...")
if namespace_exists(namespace):
    ok(f"Namespace '{namespace}' exists")
else:
    raise RuntimeError(f"❌ Namespace '{namespace}' does not exist. Complete Module 1 first.")

# Check Helm release
print(f"\n4. Checking Helm release...")
helm_release = os.environ.get("HELM_RELEASE", "langsmith")
result = run(
    ["helm", "list", "-n", namespace, "--output", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    try:
        releases = json.loads(result.stdout)
        release_names = [r.get("name") for r in releases]
        if helm_release in release_names:
            ok(f"Helm release '{helm_release}' exists")
            # Get release info
            release_info = [r for r in releases if r.get("name") == helm_release][0]
            print(f"   Status: {release_info.get('status', 'unknown')}")
            print(f"   Chart: {release_info.get('chart', 'unknown')}")
        else:
            raise RuntimeError(f"❌ Helm release '{helm_release}' not found in namespace '{namespace}'")
    except json.JSONDecodeError:
        warn("Could not parse Helm release list")
else:
    warn("Could not check Helm releases")
    print("   💡 Ensure Helm is installed and configured")

ok("Preflight checks complete")


## 3. Inspect Current Auth Configuration

Examine the current authentication configuration without leaking secrets.


In [ ]:
print("### Inspecting Current Auth Configuration\n")

# Check for auth-related environment variables in deployments
print("1. Checking deployment environment variables...")
result = run(
    ["kubectl", "get", "deployments", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    deployments = json.loads(result.stdout)
    auth_vars_found = False
    
    for deployment in deployments.get("items", []):
        name = deployment.get("metadata", {}).get("name", "")
        containers = deployment.get("spec", {}).get("template", {}).get("spec", {}).get("containers", [])
        
        for container in containers:
            env_vars = container.get("env", [])
            auth_env = [e for e in env_vars if any(keyword in e.get("name", "").upper() for keyword in ["AUTH", "OIDC", "OIDC", "SSO"])]
            
            if auth_env:
                auth_vars_found = True
                print(f"\n   Deployment: {name}")
                print(f"   Container: {container.get('name', 'default')}")
                print("   Auth-related environment variables:")
                for env in auth_env:
                    env_name = env.get("name", "")
                    # Never print secret values
                    if "SECRET" in env_name.upper() or "PASSWORD" in env_name.upper() or "TOKEN" in env_name.upper():
                        print(f"     - {env_name}: <redacted>")
                    elif env.get("value"):
                        value = env.get("value", "")
                        # Redact long values that might be secrets
                        if len(value) > 20:
                            print(f"     - {env_name}: {redact(value)}")
                        else:
                            print(f"     - {env_name}: {value}")
                    elif env.get("valueFrom"):
                        print(f"     - {env_name}: <from secret/configmap>")
    
    if not auth_vars_found:
        warn("No auth-related environment variables found in deployments")
        print("   💡 Auth may be configured via Helm values or ConfigMaps")

# Check for auth-related secrets (names only, never values)
print("\n2. Checking for auth-related secrets...")
result = run(
    ["kubectl", "get", "secrets", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    secrets = json.loads(result.stdout)
    auth_secrets = [s for s in secrets.get("items", []) 
                   if any(keyword in s.get("metadata", {}).get("name", "").lower() 
                         for keyword in ["auth", "oidc", "sso", "oauth"])]
    
    if auth_secrets:
        ok(f"Found {len(auth_secrets)} auth-related secret(s)")
        for secret in auth_secrets:
            name = secret.get("metadata", {}).get("name", "")
            print(f"   - {name} (values not displayed)")
    else:
        warn("No auth-related secrets found")
        print("   💡 Client secrets may be stored elsewhere or not yet configured")

# Try to get Helm values (if accessible)
print("\n3. Checking Helm values...")
helm_release = os.environ.get("HELM_RELEASE", "langsmith")
result = run(
    ["helm", "get", "values", helm_release, "-n", namespace, "--output", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    try:
        values = json.loads(result.stdout)
        # Look for auth configuration
        if "auth" in str(values).lower() or "oidc" in str(values).lower():
            ok("Helm values contain auth configuration")
            print("   💡 Review Helm values file for complete auth configuration")
            # Don't print values directly as they may contain secrets
        else:
            warn("No auth configuration found in Helm values")
    except json.JSONDecodeError:
        warn("Could not parse Helm values")
else:
    warn("Could not retrieve Helm values")
    print("   💡 Auth may be configured via environment variables or ConfigMaps")

ok("Auth configuration inspection complete (no secrets displayed)")


## 4. Validate Ingress/TLS Preconditions

Verify domain resolution, HTTPS accessibility, and TLS certificate validity.


In [ ]:
import socket
import ssl
import requests
from urllib.parse import urlparse
from shared._shell import run

domain = config["LANGSMITH_DOMAIN"]
print(f"### Validating Ingress/TLS for {domain}\n")

# 1. DNS Resolution
print("1. Checking DNS resolution...")
try:
    ip_address = socket.gethostbyname(domain)
    ok(f"Domain resolves to: {ip_address}")
except socket.gaierror as e:
    raise RuntimeError(f"❌ DNS resolution failed for {domain}: {e}")

# 2. HTTPS Reachability
print(f"\n2. Checking HTTPS reachability...")
https_url = f"https://{domain}"

try:
    response = requests.get(https_url, timeout=10, verify=True, allow_redirects=True)
    ok(f"HTTPS accessible: {response.status_code}")
    print(f"   Final URL: {response.url}")
except requests.exceptions.SSLError as e:
    warn(f"SSL verification failed: {e}")
    print("   💡 Certificate may be self-signed or invalid")
    print("   💡 For production, use a valid certificate from a trusted CA")
except requests.exceptions.ConnectionError as e:
    raise RuntimeError(f"❌ Could not connect to {domain}: {e}")
except requests.exceptions.Timeout:
    raise RuntimeError(f"❌ Connection to {domain} timed out")

# 3. TLS Certificate Check
print(f"\n3. Checking TLS certificate...")
try:
    context = ssl.create_default_context()
    with socket.create_connection((domain, 443), timeout=10) as sock:
        with context.wrap_socket(sock, server_hostname=domain) as ssock:
            cert = ssock.getpeercert()
            subject = dict(x[0] for x in cert['subject'])
            issuer = dict(x[0] for x in cert['issuer'])
            
            print(f"   Subject: {subject.get('commonName', 'N/A')}")
            print(f"   Issuer: {issuer.get('commonName', 'N/A')}")
            
            # Check certificate expiration
            import datetime
            not_after = datetime.datetime.strptime(cert['notAfter'], '%b %d %H:%M:%S %Y %Z')
            days_until_expiry = (not_after - datetime.datetime.now()).days
            
            if days_until_expiry > 30:
                ok(f"Certificate valid for {days_until_expiry} more days")
            elif days_until_expiry > 0:
                warn(f"Certificate expires in {days_until_expiry} days")
            else:
                raise RuntimeError(f"❌ Certificate expired {abs(days_until_expiry)} days ago")
            
            # Check if certificate matches domain
            cert_domains = []
            if 'subjectAltName' in cert:
                cert_domains = [name[1] for name in cert['subjectAltName'] if name[0] == 'DNS']
            elif 'commonName' in subject:
                cert_domains = [subject['commonName']]
            
            if domain in cert_domains or any(domain.endswith(f".{d}") or d == f"*.{domain.split('.', 1)[1]}" for d in cert_domains):
                ok("Certificate matches domain")
            else:
                warn(f"Certificate domains {cert_domains} may not match {domain}")
                
except Exception as e:
    warn(f"Could not verify TLS certificate: {e}")
    print("   💡 Certificate validation failed, but connection may still work")

# 4. Check Ingress Configuration
print(f"\n4. Checking ingress configuration...")
result = run(
    ["kubectl", "get", "ingress", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    ingresses = json.loads(result.stdout)
    matching_ingress = None
    
    for ingress in ingresses.get("items", []):
        rules = ingress.get("spec", {}).get("rules", [])
        for rule in rules:
            host = rule.get("host", "")
            if domain in host or host == domain:
                matching_ingress = ingress
                break
    
    if matching_ingress:
        ok(f"Ingress found for domain")
        name = matching_ingress.get("metadata", {}).get("name", "")
        print(f"   Ingress name: {name}")
        
        # Check TLS configuration
        tls = matching_ingress.get("spec", {}).get("tls", [])
        if tls:
            ok("TLS configured in ingress")
            for tls_config in tls:
                hosts = tls_config.get("hosts", [])
                secret = tls_config.get("secretName", "")
                print(f"   Hosts: {', '.join(hosts)}")
                print(f"   Secret: {secret}")
        else:
            warn("No TLS configuration found in ingress")
    else:
        warn(f"No ingress found matching domain {domain}")
else:
    warn("Could not retrieve ingress configuration")

ok("Ingress/TLS preconditions validated")


In [ ]:
from shared._shell import run

print("### Validating OIDC Settings\n")

issuer = config["OIDC_ISSUER"]
redirect_uri = config["OIDC_REDIRECT_URI"]
client_id = config["OIDC_CLIENT_ID"]

# 1. Issuer URL Reachability
print("1. Checking issuer URL reachability...")
if not issuer.startswith("https://"):
    raise RuntimeError(f"❌ Issuer URL must use HTTPS: {issuer}")

# Try to fetch .well-known/openid-configuration
well_known_url = f"{issuer.rstrip('/')}/.well-known/openid-configuration"

try:
    response = requests.get(well_known_url, timeout=10, verify=True)
    if response.status_code == 200:
        ok("Issuer discovery endpoint accessible")
        try:
            discovery = response.json()
            print(f"   Authorization endpoint: {discovery.get('authorization_endpoint', 'N/A')}")
            print(f"   Token endpoint: {discovery.get('token_endpoint', 'N/A')}")
            print(f"   Userinfo endpoint: {discovery.get('userinfo_endpoint', 'N/A')}")
            
            # Validate issuer matches
            discovered_issuer = discovery.get("issuer", "")
            if discovered_issuer == issuer or discovered_issuer.rstrip("/") == issuer.rstrip("/"):
                ok("Discovered issuer matches configuration")
            else:
                warn(f"Issuer mismatch: config={issuer}, discovered={discovered_issuer}")
                print("   💡 Update OIDC_ISSUER to match discovered issuer")
        except json.JSONDecodeError:
            warn("Discovery endpoint returned invalid JSON")
    else:
        warn(f"Issuer discovery endpoint returned {response.status_code}")
        print("   💡 Issuer may not support discovery, or URL is incorrect")
except requests.exceptions.RequestException as e:
    warn(f"Could not reach issuer discovery endpoint: {e}")
    print("   💡 Verify issuer URL is correct and accessible from this network")

# 2. Redirect URI Exactness
print(f"\n2. Validating redirect URI exactness...")
print(f"   Configured: {redirect_uri}")

# Parse redirect URI
parsed = urlparse(redirect_uri)
if parsed.scheme != "https":
    warn("Redirect URI should use HTTPS in production")

if parsed.netloc != domain:
    warn(f"Redirect URI domain '{parsed.netloc}' does not match LANGSMITH_DOMAIN '{domain}'")

# Check for common mistakes
if redirect_uri.endswith("/") and not redirect_uri.endswith("/auth/callback/"):
    warn("Redirect URI has trailing slash - ensure IdP whitelist matches exactly")

if " " in redirect_uri:
    raise RuntimeError("❌ Redirect URI contains spaces - this is invalid")

ok("Redirect URI format validated")
print("   💡 Ensure this EXACT value is whitelisted in your IdP")

# 3. Required Claims Mapping
print(f"\n3. Validating claims mapping...")
print(f"   Email claim: {config['OIDC_EMAIL_CLAIM']}")
print(f"   Name claim: {config['OIDC_NAME_CLAIM']}")
print(f"   Groups claim: {config['OIDC_GROUPS_CLAIM']}")

if not config.get("OIDC_EMAIL_CLAIM"):
    raise RuntimeError("❌ Email claim mapping is required")

ok("Claims mapping configured")
print("   💡 Verify your IdP sends these claims in the token")

# 4. Scopes Validation
print(f"\n4. Validating scopes...")
scopes = config.get("OIDC_SCOPES", "").split(",")
scopes = [s.strip() for s in scopes]

required_scopes = ["openid", "email"]
for scope in required_scopes:
    if scope not in scopes:
        warn(f"Required scope '{scope}' not found in configuration")

print(f"   Configured scopes: {', '.join(scopes)}")
if "groups" in scopes:
    ok("Groups scope configured (required for group-based role mapping)")
else:
    warn("Groups scope not configured - group-based role mapping will not work")

ok("OIDC settings validation complete")


## 6. Deployment Verification

Verify pods are ready, logs show auth provider configured, and endpoints return expected auth behavior.


In [ ]:
from shared._k8s_helpers import get_pods, wait_for_deployments_ready, require_namespace
from shared._shell import run

print("### Deployment Verification\n")

# 1. Pod Readiness
print("1. Checking pod readiness...")
require_namespace(namespace)

try:
    wait_for_deployments_ready(namespace, timeout="5m")
    ok("All deployments ready")
except Exception as e:
    warn(f"Some deployments may not be ready: {e}")
    print("   💡 Check pod status manually: kubectl get pods -n {namespace}")

# Get pod status
pods_output = get_pods(namespace)
print("\nPod Status:")
print(pods_output)

# 2. Check Logs for Auth Configuration
print("\n2. Checking logs for auth provider configuration...")
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "jsonpath={.items[*].metadata.name}"],
    check=False,
    stream=False
)

if result.returncode == 0 and result.stdout.strip():
    pod_names = result.stdout.strip().split()
    # Look for API/server pods (most likely to have auth config)
    api_pods = [p for p in pod_names if any(keyword in p.lower() for keyword in ["api", "server", "backend"])]
    
    if not api_pods:
        api_pods = pod_names[:2]  # Fallback to first 2 pods
    
    auth_configured = False
    for pod_name in api_pods[:2]:  # Check up to 2 pods
        try:
            # Get recent logs (last 100 lines)
            log_result = run(
                ["kubectl", "logs", pod_name, "-n", namespace, "--tail=100"],
                check=False,
                stream=False
            )
            
            if log_result.returncode == 0:
                logs = log_result.stdout.lower()
                # Look for auth-related log messages
                auth_indicators = [
                    "oidc",
                    "oauth",
                    "auth provider",
                    "authentication",
                    "sso",
                    "issuer",
                ]
                
                found_indicators = [ind for ind in auth_indicators if ind in logs]
                if found_indicators:
                    auth_configured = True
                    print(f"   Pod {pod_name}: Auth indicators found")
                    # Don't print log lines as they may contain sensitive info
        except Exception:
            pass
    
    if auth_configured:
        ok("Auth provider appears configured in logs")
    else:
        warn("No clear auth configuration indicators in logs")
        print("   💡 This may be normal if auth is configured but not yet used")
else:
    warn("Could not retrieve pod names")

# 3. Test Endpoint Auth Behavior
print(f"\n3. Testing endpoint auth behavior...")
test_url = f"https://{domain}/api/v1/me"  # Common auth-check endpoint

try:
    # Unauthenticated request should redirect or return 401/403
    response = requests.get(test_url, timeout=10, verify=True, allow_redirects=False)
    
    if response.status_code == 401:
        ok("Endpoint requires authentication (401)")
    elif response.status_code == 403:
        ok("Endpoint requires authorization (403)")
    elif response.status_code in [301, 302, 307, 308]:
        redirect_location = response.headers.get("Location", "")
        if "login" in redirect_location.lower() or "auth" in redirect_location.lower():
            ok("Endpoint redirects to authentication")
            print(f"   Redirect location: {redirect_location}")
        else:
            warn(f"Endpoint redirects but not to auth: {redirect_location}")
    else:
        warn(f"Unexpected status code: {response.status_code}")
        print("   💡 Endpoint may be publicly accessible or auth not configured")
except requests.exceptions.SSLError:
    warn("SSL error (may be expected with self-signed certs)")
except requests.exceptions.RequestException as e:
    warn(f"Could not test endpoint: {e}")

# 4. Check for Auth-related ConfigMaps
print(f"\n4. Checking for auth-related ConfigMaps...")
result = run(
    ["kubectl", "get", "configmaps", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    configmaps = json.loads(result.stdout)
    auth_configmaps = [cm for cm in configmaps.get("items", [])
                      if any(keyword in cm.get("metadata", {}).get("name", "").lower()
                            for keyword in ["auth", "oidc", "sso"])]
    
    if auth_configmaps:
        ok(f"Found {len(auth_configmaps)} auth-related ConfigMap(s)")
        for cm in auth_configmaps:
            name = cm.get("metadata", {}).get("name", "")
            print(f"   - {name}")
    else:
        print("   No auth-related ConfigMaps found (may use environment variables)")

ok("Deployment verification complete")


## 7. Failure Drills (Optional)

**⚠️ WARNING:** These drills modify configuration and may break authentication. Only run in non-production environments with explicit opt-in.

These drills help you understand failure modes and recovery procedures. Each drill shows:
- What to change
- Expected failure behavior
- How to revert
- Validation commands


In [ ]:
print("### Failure Drills (Opt-In Only)\n")

enable_drills = os.environ.get("ENABLE_FAILURE_DRILLS", "false").lower() == "true"

if not enable_drills:
    print("⚠️  Failure drills are DISABLED by default.")
    print("   To enable, set ENABLE_FAILURE_DRILLS=true in your environment")
    print("\n   These drills will:")
    print("   1. Modify Helm values or environment variables")
    print("   2. Restart pods to apply changes")
    print("   3. Test authentication failures")
    print("   4. Revert changes")
    print("\n   💡 Only run in non-production environments!")
    print("\n   Available drills:")
    print("   - Drill 1: Redirect URI mismatch")
    print("   - Drill 2: Missing required claim")
    print("   - Drill 3: Invalid client secret")
else:
    print("⚠️  FAILURE DRILLS ENABLED")
    print("   Proceeding with guided failure drills...\n")
    
    # Drill 1: Redirect URI Mismatch
    print("=" * 60)
    print("DRILL 1: Redirect URI Mismatch")
    print("=" * 60)
    print("\nThis drill simulates a redirect URI mismatch between LangSmith and IdP.")
    print("\nSteps:")
    print("1. Modify redirect URI in Helm values (add trailing slash)")
    print("2. Restart pods")
    print("3. Attempt login (should fail)")
    print("4. Revert change")
    print("5. Restart pods")
    print("6. Verify login works again")
    print("\n💡 To perform this drill manually:")
    print(f"   1. Edit Helm values: redirect URI = '{redirect_uri}/'")
    print("   2. helm upgrade <release> <chart> -n <namespace> -f <values-file>")
    print("   3. kubectl rollout restart deployment -n <namespace>")
    print("   4. Test login (should fail with redirect mismatch)")
    print("   5. Revert values file")
    print("   6. helm upgrade again")
    print("   7. kubectl rollout restart again")
    print("   8. Test login (should work)")
    
    # Drill 2: Missing Required Claim
    print("\n" + "=" * 60)
    print("DRILL 2: Missing Required Claim")
    print("=" * 60)
    print("\nThis drill simulates missing email claim in token.")
    print("\nSteps:")
    print("1. Temporarily remove email claim mapping")
    print("2. Restart pods")
    print("3. Attempt login (should fail or user has no email)")
    print("4. Restore claim mapping")
    print("5. Restart pods")
    print("6. Verify login works")
    print("\n💡 To perform this drill manually:")
    print("   1. Remove OIDC_EMAIL_CLAIM from environment or Helm values")
    print("   2. Restart pods")
    print("   3. Test login")
    print("   4. Restore OIDC_EMAIL_CLAIM")
    print("   5. Restart pods")
    print("   6. Test login")
    
    # Drill 3: Invalid Client Secret
    print("\n" + "=" * 60)
    print("DRILL 3: Invalid Client Secret")
    print("=" * 60)
    print("\nThis drill simulates client secret rotation gone wrong.")
    print("\nSteps:")
    print("1. Update client secret to invalid value")
    print("2. Restart pods")
    print("3. Attempt login (should fail with 'invalid client')")
    print("4. Restore correct secret")
    print("5. Restart pods")
    print("6. Verify login works")
    print("\n💡 To perform this drill manually:")
    print("   1. kubectl create secret generic langsmith-oidc-secret \\")
    print("        --from-literal=client-secret=INVALID_SECRET -n <namespace> --dry-run=client -o yaml | kubectl apply -f -")
    print("   2. Restart pods")
    print("   3. Test login (should fail)")
    print("   4. Restore correct secret from IdP")
    print("   5. Restart pods")
    print("   6. Test login")

print("\n" + "=" * 60)
print("Validation Commands After Each Drill:")
print("=" * 60)
print("\n1. Check pod logs for errors:")
print("   kubectl logs <pod-name> -n <namespace> --tail=50 | grep -i auth")
print("\n2. Test endpoint response:")
print(f"   curl -I https://{domain}/api/v1/me")
print("\n3. Check Helm values:")
print("   helm get values <release> -n <namespace>")
print("\n4. Verify environment variables:")
print("   kubectl exec <pod-name> -n <namespace> -- env | grep -i oidc")


## 8. Support Bundle Pointers

When troubleshooting authentication issues, collect these artifacts for support.


In [ ]:
from datetime import datetime
import re
from shared._shell import run

print("### Support Bundle Collection\n")

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
support_dir = artifacts_dir / f"auth-support-{timestamp}"
support_dir.mkdir(exist_ok=True)

print(f"Saving support artifacts to: {support_dir}\n")

# 1. Pod Logs
print("1. Collecting pod logs...")
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "jsonpath={.items[*].metadata.name}"],
    check=False,
    stream=False
)

if result.returncode == 0 and result.stdout.strip():
    pod_names = result.stdout.strip().split()
    api_pods = [p for p in pod_names if any(keyword in p.lower() for keyword in ["api", "server", "backend"])]
    
    for pod_name in (api_pods[:3] if api_pods else pod_names[:3]):
        try:
            log_result = run(
                ["kubectl", "logs", pod_name, "-n", namespace, "--tail=200"],
                check=False,
                stream=False
            )
            if log_result.returncode == 0:
                log_file = support_dir / f"{pod_name}-logs.txt"
                with open(log_file, "w") as f:
                    f.write(log_result.stdout)
                    if log_result.stderr:
                        f.write("\n\nSTDERR:\n")
                        f.write(log_result.stderr)
                print(f"   ✅ Saved logs for {pod_name}")
        except Exception as e:
            print(f"   ⚠️  Could not save logs for {pod_name}: {e}")

# 2. Deployment Configuration (redacted)
print("\n2. Collecting deployment configuration...")
result = run(
    ["kubectl", "get", "deployments", "-n", namespace, "-o", "yaml"],
    check=False,
    stream=False
)

if result.returncode == 0:
    # Redact secrets from YAML
    yaml_content = result.stdout
    # Simple redaction - replace secret values
    import re
    # This is a simple approach - in production, use proper YAML parsing
    redacted_yaml = re.sub(r'(value:)\s+[^\n]{20,}', r'\1 <redacted>', yaml_content)
    
    with open(support_dir / "deployments.yaml", "w") as f:
        f.write(redacted_yaml)
    print("   ✅ Saved deployment configuration (secrets redacted)")

# 3. Ingress Configuration
print("\n3. Collecting ingress configuration...")
result = run(
    ["kubectl", "get", "ingress", "-n", namespace, "-o", "yaml"],
    check=False,
    stream=False
)

if result.returncode == 0:
    with open(support_dir / "ingress.yaml", "w") as f:
        f.write(result.stdout)
    print("   ✅ Saved ingress configuration")

# 4. Helm Values (if accessible)
print("\n4. Collecting Helm values...")
helm_release = os.environ.get("HELM_RELEASE", "langsmith")
result = run(
    ["helm", "get", "values", helm_release, "-n", namespace],
    check=False,
    stream=False
)

if result.returncode == 0:
    # Redact secrets
    values_content = result.stdout
    redacted_values = re.sub(r'(secret|password|token|clientSecret):\s+[^\n]+', r'\1: <redacted>', values_content, flags=re.IGNORECASE)
    
    with open(support_dir / "helm-values.txt", "w") as f:
        f.write(redacted_values)
    print("   ✅ Saved Helm values (secrets redacted)")

# 5. Events
print("\n5. Collecting recent events...")
result = run(
    ["kubectl", "get", "events", "-n", namespace, "--sort-by=.lastTimestamp"],
    check=False,
    stream=False
)

if result.returncode == 0:
    with open(support_dir / "events.txt", "w") as f:
        f.write(result.stdout)
    print("   ✅ Saved events")

# 6. Configuration Summary (redacted)
print("\n6. Creating configuration summary...")
summary = f"""Auth Configuration Summary
Generated: {datetime.now().isoformat()}

Domain: {domain}
Namespace: {namespace}
Issuer: {issuer}
Client ID: {client_id}
Redirect URI: {redirect_uri}
Email Claim: {config['OIDC_EMAIL_CLAIM']}
Name Claim: {config['OIDC_NAME_CLAIM']}
Groups Claim: {config['OIDC_GROUPS_CLAIM']}
Scopes: {config.get('OIDC_SCOPES', 'N/A')}

Note: Client secret is not included for security.
"""

with open(support_dir / "config-summary.txt", "w") as f:
    f.write(summary)
print("   ✅ Saved configuration summary (secrets redacted)")

ok(f"Support bundle saved to: {support_dir}")
print("\n💡 Include these files when contacting support:")
print("   - Pod logs (especially API/server pods)")
print("   - Configuration summary")
print("   - Helm values (if accessible)")
print("   - Recent events")
print("\n💡 Do NOT include:")
print("   - Client secrets")
print("   - Tokens")
print("   - Passwords")
print("\n💡 For comprehensive debugging, see:")
print("   - docs/shared/auth_troubleshooting.md")
print("   - Your IdP's authentication logs")


## Summary

### ✅ Validation Complete

This notebook has validated:
- ✅ Environment configuration loaded (secrets redacted)
- ✅ Preflight checks passed
- ✅ Current auth configuration inspected
- ✅ Ingress/TLS preconditions verified
- ✅ OIDC settings validated
- ✅ Deployment verified
- ✅ Support bundle collected

### 🎯 Next Steps

1. **Complete the validation checklist:**
   - See `docs/shared/auth_validation_checklist.md`
   - Test login with admin user
   - Test login with standard user
   - Verify role mapping
   - Test logout

2. **Review troubleshooting guide:**
   - See `docs/shared/auth_troubleshooting.md`
   - Bookmark for future reference

3. **Document your configuration:**
   - Save Helm values (with secrets redacted)
   - Document IdP settings
   - Record group-to-role mappings

4. **Proceed to Module 3** (if applicable)

### 📋 Common Issues

If validation failed, check:
- Redirect URI matches exactly between LangSmith and IdP
- Client secret is correct and not expired
- Required claims are being sent by IdP
- TLS certificate is valid and trusted
- Network connectivity to IdP issuer URL

See `docs/shared/auth_troubleshooting.md` for detailed troubleshooting steps.
